# 06 — Normalization: keeping activations healthy during training

In notebook 4 you chose the initial weight variance so that, *at the start*, the signal keeps a
healthy size as it flows through a deep network. But the weights move during training — and as they
move, the per-layer distributions drift again: they re-shrink toward zero, re-saturate, or re-explode,
and the vanishing/exploding trouble of notebook 3 creeps back. **Normalization** answers with a
simple, powerful move: instead of only setting the starting variance, **re-normalize a layer's
activations at every step**. We build the most common form — **batch normalization** — by hand, as a
complete layer with both a forward and a backward pass, and watch what it does.

**Prerequisites**
- Notebook 3 — vanishing/exploding gradients, caused by activations that drift across depth.
- Notebook 4 — initialization (He / Xavier) sets the activation variance *at initialization*.
- Notebooks 1–5 — the `L`-layer network (forward, backward, training loop), ReLU, mini-batch SGD.

**What you'll be able to do**
- Implement a **batch-norm layer** (forward and backward) and gradient-check it.
- Show it pin every layer's activations to a healthy band, *whatever* the initialization.
- Train a deep network that had stalled at chance — and see where the gain is real and where it isn't.
- Meet **layer normalization** (the transformer's choice) and name what the library's `nn.BatchNorm` adds.

This is the **last notebook we build by hand in numpy**. Notebook 7 moves the very same network to PyTorch.

## The problem normalization solves

Initialization (notebook 4) is a one-time choice: pick the weight variance so that, on the *first*
forward pass, each layer's activations have a sensible spread. He initialization does this well for
ReLU. But training then changes every weight. A few hundred steps later the carefully chosen variance
is gone, and the per-layer distributions are free to drift — back toward the dying or exploding regimes
of notebook 3.

The idea of normalization is to stop relying on a lucky starting point and instead **re-center and
re-scale the activations at every step**. It is the same lever as initialization — control the variance
as the signal moves through depth — but applied *throughout* training, not only at the beginning.

## The mechanism: batch normalization

Take the pre-activations `z` of one layer over the current **mini-batch** — a matrix with one row per
sample and one column per unit. For each unit (each column), compute the **mean** and **variance over
the batch**, then standardize:

`ẑ = (z − μ) / √(σ² + ε)`

so every unit now has mean 0 and variance 1 across the batch (`ε` is a tiny constant that avoids
dividing by zero). Forcing every layer to be exactly mean-0/variance-1 would be too rigid, though — a
unit might *need* a shifted or stretched distribution. So batch norm hands control back with two
**learnable** parameters per unit: a scale **γ** and a shift **β**:

`out = γ · ẑ + β`

With `γ = 1, β = 0` this is plain standardization; training is free to adapt γ and β — and, if it
helps, to undo the normalization entirely. This is the layer Ioffe and Szegedy introduced in 2015.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_circles
from sklearn.preprocessing import StandardScaler

from ml_course import colors, viz

viz.use_course_style()

SEED = 0
EPS = 1e-5


def bn_forward(z, gamma, beta, eps=EPS):
    # Standardize each feature across the batch (axis 0), then scale + shift.
    mu = z.mean(axis=0)
    var = z.var(axis=0)                 # population variance (ddof=0)
    istd = 1.0 / np.sqrt(var + eps)
    zhat = (z - mu) * istd
    out = gamma * zhat + beta
    return out, (zhat, istd, gamma)     # cache for the backward pass


# Two badly-scaled features: one tight around 5, one wide around -3.
rng = np.random.default_rng(SEED)
z = np.column_stack([rng.normal(5.0, 0.1, size=512), rng.normal(-3.0, 8.0, size=512)])
d = z.shape[1]
out, _ = bn_forward(z, gamma=np.ones(d), beta=np.zeros(d))
print(f"{'':10}{'feature 0':>22}{'feature 1':>22}")
print(f"{'before':10}  mean {z[:, 0].mean():+7.3f} std {z[:, 0].std():6.3f}"
      f"     mean {z[:, 1].mean():+7.3f} std {z[:, 1].std():6.3f}")
print(f"{'after BN':10}  mean {out[:, 0].mean():+7.3f} std {out[:, 0].std():6.3f}"
      f"     mean {out[:, 1].mean():+7.3f} std {out[:, 1].std():6.3f}")

In [ ]:
# A schematic of the batch-norm layer: standardize per feature across the batch, then scale + shift.
fig, ax = plt.subplots(figsize=(12.5, 4.4))
ax.axis("off")
ax.set_xlim(0, 12.5)
ax.set_ylim(0, 4.4)


def box(x, text, face):
    ax.text(x, 2.2, text, ha="center", va="center", fontsize=11, color=colors.COLORS["text"],
            bbox=dict(boxstyle="round,pad=0.55", facecolor=face, edgecolor=colors.COLORS["text"],
                      linewidth=1.0))


def arrow(x0, x1):
    ax.annotate("", xy=(x1, 2.2), xytext=(x0, 2.2),
                arrowprops=dict(arrowstyle="-|>", color=colors.COLORS["text"], lw=1.6))


box(2.0, "a mini-batch of\npre-activations  z\n(n samples × d units)", colors.COLORS["muted"])
arrow(3.7, 5.0)
norm_txt = "normalize each feature\nacross the batch\nẑ = (z − μ) / √(σ² + ε)"
box(6.4, norm_txt, colors.COLORS["model"])
arrow(8.1, 9.0)
box(10.6, "scale & shift\nout = γ · ẑ + β\n(γ, β learnable)", colors.COLORS["correct"])

ax.text(6.4, 0.7, "μ, σ² are computed down each column — across the n samples of the batch",
        ha="center", va="center", fontsize=10, color=colors.COLORS["highlight"])
ax.text(10.6, 0.7, "γ = 1, β = 0 → plain standardization;\ntraining adapts them",
        ha="center", va="center", fontsize=9, color=colors.COLORS["text"])
ax.set_title("The batch-norm layer", color=colors.COLORS["text"])
fig.tight_layout()
plt.show()

**Read the figure.** A batch-norm layer takes a mini-batch of pre-activations and does three things.
First it measures, **for each unit separately**, the mean and variance **down its column** — that is,
across the `n` samples in the batch (this is what "batch" norm means). Then it standardizes each unit
to mean 0, variance 1. Finally it applies the two learnable knobs, `γ` (scale) and `β` (shift): with
`γ = 1, β = 0` the output is plain standardization, and training is free to move them — even back to the
original distribution if that serves the loss. The whole transform is differentiable, so γ and β train
by gradient descent like any other parameter.

## Watch it fix the drift

Notebook 4 measured what happens to a 10-layer ReLU stack's activations under different initializations:
with a too-small scale they **die** toward zero; with a too-large scale they **explode**; with He they
are preserved — *at the start*. Now we rebuild that stack (here at width 32), insert a batch-norm layer
after each linear map, and measure the per-layer activation standard deviation again, for all three
initializations.

In [ ]:
def init_params(sizes, kind, seed=SEED):
    rng = np.random.default_rng(seed)
    params = []
    for a, b in zip(sizes[:-1], sizes[1:], strict=True):
        if kind == "small":
            s = 0.1
        elif kind == "large":
            s = 1.0
        else:                            # "he": std = sqrt(2 / fan_in)
            s = np.sqrt(2.0 / a)
        params.append({"W": rng.standard_normal((a, b)) * s, "b": np.zeros(b)})
    return params


def act(z, name):
    if name == "sigmoid":
        return 1.0 / (1.0 + np.exp(-z))
    if name == "tanh":
        return np.tanh(z)
    return np.maximum(0.0, z)            # relu


def dact(a, name):
    if name == "sigmoid":
        return a * (1.0 - a)
    if name == "tanh":
        return 1.0 - a ** 2
    return (a > 0).astype(float)         # relu'


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def ce_loss(probs, y):
    return -np.log(probs[np.arange(len(y)), y] + 1e-12).mean()


def make_bn(sizes, use_bn):
    # One {gamma, beta} per HIDDEN layer; the softmax output layer is never normalized.
    bn = [None] * (len(sizes) - 1)
    if use_bn:
        for i in range(len(sizes) - 2):
            bn[i] = {"gamma": np.ones(sizes[i + 1]), "beta": np.zeros(sizes[i + 1])}
    return bn


def net_forward(params, bn, X, hidden):
    a, last = X, len(params) - 1
    acts, caches = [X], []
    for i, layer in enumerate(params):
        z = a @ layer["W"] + layer["b"]
        if i == last:
            a, c = softmax(z), None                  # output: no BN, no activation derivative
        else:
            if bn[i] is not None:
                u, c = bn_forward(z, bn[i]["gamma"], bn[i]["beta"])
            else:
                u, c = z, None
            a = act(u, hidden)
        acts.append(a)
        caches.append(c)
    return acts, caches


# A 10-layer ReLU stack on the circles data; measure each hidden layer's activation std.
X, y = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=SEED)
X = StandardScaler().fit_transform(X)
stack = [2] + [32] * 10 + [2]


def layer_stds(kind, use_bn):
    p = init_params(stack, kind)
    acts, _ = net_forward(p, make_bn(stack, use_bn), X, "relu")
    return [acts[i].std() for i in range(1, len(p))]   # the 10 hidden layers


print(f"{'init':8}{'BN?':5}{'layer 1':>10}{'layer 5':>12}{'layer 10':>12}")
for kind in ("small", "large", "he"):
    for use_bn in (False, True):
        s = layer_stds(kind, use_bn)
        print(f"{kind:8}{('yes' if use_bn else 'no'):5}{s[0]:10.3f}{s[4]:12.3f}{s[9]:12.3f}")

In [ ]:
layers = np.arange(1, 11)
fig, (axl, axr) = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
palette = [colors.COLORS["class_a"], colors.COLORS["class_b"], colors.COLORS["class_c"]]

for kind, col in zip(("small", "large", "he"), palette, strict=True):
    axl.plot(layers, np.maximum(layer_stds(kind, False), 1e-4), marker="o", color=col, label=kind)
    axr.plot(layers, layer_stds(kind, True), marker="o", color=col, label=kind)

axl.set_yscale("log")
axl.set_title("Without BN: the activation scale is at the mercy of the init")
axl.set_xlabel("hidden layer")
axl.set_ylabel("activation std (log scale)")
axl.legend(title="init")

axr.set_ylim(0.0, 1.15)
axr.set_title("With BN: one flat, healthy band — whatever the init")
axr.set_xlabel("hidden layer")
axr.set_ylabel("activation std")
axr.legend(title="init")
fig.tight_layout()
plt.show()

**Read the figure.** Left, without batch norm, the per-layer activation standard deviation is decided
entirely by the initialization (note the **log scale**): the small init **vanishes** toward zero
(floored at `1e-4` here so it stays on the log axis), the large init **explodes** to about `10⁵` by
layer ten (the table in the previous cell reads `128 542`), and only He sits in a usable range — and
only because we chose it. Right, with batch norm inserted after every linear map, all three
initializations **collapse onto the same flat band** (around `0.6`) across all ten layers. Batch norm
pins each layer's *pre-activation* to variance 1; after ReLU keeps only the positive half, the standard
deviation settles near `0.6`, and — the point — it no longer drifts with depth. The network's health no longer depends on a lucky starting scale.

## Wiring batch norm into training — the backward pass

In a network, batch norm sits between the linear map and the activation: `z = a·W + b → BN → ReLU`. One
consequence is worth noting: because BN subtracts the mean, adding a constant bias `b` before it has no
effect (the mean absorbs it), so **the pre-BN bias becomes redundant — β is the effective bias**. (This
is why, in libraries, a linear layer that feeds a batch-norm layer is built with `bias=False`.)

To train γ and β we need the gradient of the loss through the batch-norm transform. Differentiating
`out = γ·ẑ + β` gives `dβ = Σ dout` and `dγ = Σ (dout · ẑ)`; propagating back through the standardization
(which couples all the samples, since μ and σ² depend on the whole batch) gives

`dz = (1/n) · istd · ( n·dẑ − Σ dẑ − ẑ · Σ(dẑ · ẑ) )`,  with `dẑ = dout · γ`.

We will trust this only after checking it against finite differences.

In [ ]:
def bn_backward(dout, cache):
    zhat, istd, gamma = cache
    n = dout.shape[0]
    dbeta = dout.sum(axis=0)
    dgamma = (dout * zhat).sum(axis=0)
    dzhat = dout * gamma
    dz = (1.0 / n) * istd * (n * dzhat - dzhat.sum(axis=0) - zhat * (dzhat * zhat).sum(axis=0))
    return dz, dgamma, dbeta


def net_backward(params, bn, acts, caches, y, hidden):
    n, L = len(y), len(params)
    yoh = np.zeros_like(acts[-1])
    yoh[np.arange(n), y] = 1.0
    gW = [None] * L
    gbn = [None] * L
    dz = (acts[-1] - yoh) / n                        # softmax + cross-entropy gradient
    for i in reversed(range(L)):
        gW[i] = {"W": acts[i].T @ dz, "b": dz.sum(axis=0)}
        if i > 0:
            du = (dz @ params[i]["W"].T) * dact(acts[i], hidden)
            if bn[i - 1] is not None:
                dz, dg, db = bn_backward(du, caches[i - 1])
                gbn[i - 1] = {"gamma": dg, "beta": db}
            else:
                dz = du
    return gW, gbn


# Gradient-check on a small net with BN active: analytic vs central finite differences.
rng = np.random.default_rng(1)
Xc, yc = rng.standard_normal((20, 3)), rng.integers(0, 2, size=20)
pc, bnc = init_params([3, 5, 2], "he", seed=2), make_bn([3, 5, 2], True)
acts, caches = net_forward(pc, bnc, Xc, "relu")
gW, gbn = net_backward(pc, bnc, acts, caches, yc, "relu")


def numeric(arr, idx, h=1e-6):
    keep = arr[idx]
    arr[idx] = keep + h
    lp = ce_loss(net_forward(pc, bnc, Xc, "relu")[0][-1], yc)
    arr[idx] = keep - h
    lm = ce_loss(net_forward(pc, bnc, Xc, "relu")[0][-1], yc)
    arr[idx] = keep
    return (lp - lm) / (2 * h)


for name, arr, ana, idx in [("W[0][0,0]", pc[0]["W"], gW[0]["W"], (0, 0)),
                            ("gamma[0][0]", bnc[0]["gamma"], gbn[0]["gamma"], (0,)),
                            ("gamma[0][2]", bnc[0]["gamma"], gbn[0]["gamma"], (2,)),
                            ("beta[0][0]", bnc[0]["beta"], gbn[0]["beta"], (0,))]:
    num = numeric(arr, idx)
    rel = abs(num - ana[idx]) / max(abs(num) + abs(ana[idx]), 1e-12)
    print(f"  {name:12}: analytic {ana[idx]:+.6e}  numeric {num:+.6e}  rel-err {rel:.1e}")
print(f"  pre-BN bias is redundant under BN: |db[0]| analytic = {abs(gW[0]['b'][0]):.1e} (≈ 0)")

**Read the gradient-check.** For the weights and for both batch-norm parameters γ and β, the analytic
gradient matches the finite-difference estimate to roughly one part in `10⁹` — so the by-hand backward
pass is correct, and the network trains end to end with γ and β as ordinary learnable parameters. The
last line confirms the side remark from above: the gradient of the pre-BN bias `b` is essentially zero,
because batch norm re-centers the activations and β has taken over the bias role.

## The training-time effect

Now the payoff. We take a genuinely deep ReLU network (six hidden layers) on the circles data and give
it a **poor** small-scale initialization — exactly the setup that stalled at chance in notebooks 3–4.
We train it with and without batch norm. We also try a learning rate large enough to break even a
*well*-initialized (He) network, to see whether batch norm buys robustness there too.

In [ ]:
def train(sizes, hidden, kind, use_bn, lr, epochs=200, batch=64, seed=SEED):
    p = init_params(sizes, kind, seed)
    bn = make_bn(sizes, use_bn)
    rng = np.random.default_rng(seed)
    n, losses = X.shape[0], []
    for _ in range(epochs):
        order = rng.permutation(n)
        for s in range(0, n, batch):
            b = order[s:s + batch]
            acts, caches = net_forward(p, bn, X[b], hidden)
            gW, gbn = net_backward(p, bn, acts, caches, y[b], hidden)
            for i in range(len(p)):
                p[i]["W"] -= lr * gW[i]["W"]
                p[i]["b"] -= lr * gW[i]["b"]
                if bn[i] is not None:
                    bn[i]["gamma"] -= lr * gbn[i]["gamma"]
                    bn[i]["beta"] -= lr * gbn[i]["beta"]
        losses.append(ce_loss(net_forward(p, bn, X, hidden)[0][-1], y))
    acc = (net_forward(p, bn, X, hidden)[0][-1].argmax(1) == y).mean()
    return losses, acc


deep = [2] + [16] * 6 + [2]
runs = {}
print(f"{'network':28}{'none: acc':>11}{'BN: acc':>10}{'none: loss':>13}{'BN: loss':>11}")
for label, hidden, kind, lr in [("ReLU, small init, lr 0.3", "relu", "small", 0.3),
                                ("ReLU, He init,   lr 0.3", "relu", "he", 0.3),
                                ("ReLU, He init,   lr 1.0", "relu", "he", 1.0),
                                ("tanh, small init, lr 0.3", "tanh", "small", 0.3),
                                ("sigmoid, small init,lr0.3", "sigmoid", "small", 0.3)]:
    l_no, a_no = train(deep, hidden, kind, False, lr)
    l_bn, a_bn = train(deep, hidden, kind, True, lr)
    runs[label] = (l_no, l_bn)
    print(f"{label:28}{a_no:11.3f}{a_bn:10.3f}{l_no[-1]:13.4f}{l_bn[-1]:11.4f}")

In [ ]:
l_no, l_bn = runs["ReLU, small init, lr 0.3"]
ep = np.arange(1, len(l_no) + 1)

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(ep, l_no, color=colors.COLORS["error"], lw=2, label="no BN (stalled)")
ax.plot(ep, l_bn, color=colors.COLORS["model"], lw=2, label="batch norm")
ax.axhline(np.log(2), color=colors.COLORS["muted"], lw=1, ls=":", label="ln 2 = chance")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("Deep ReLU net, small init: BN turns a stalled net into one that learns")
ax.legend()
fig.tight_layout()
plt.show()

**Read the figure, and read it honestly.** Without batch norm (coral) the loss never leaves `ln 2` —
the network is stuck at chance, the notebook-3 stall, because the small init never lets a usable signal
reach the output. With batch norm (blue) the loss descends to near zero and the network reaches 100%
accuracy. The table tells the fuller story: batch norm rescues the stalled net for ReLU, tanh, *and*
sigmoid, and it trains even at a learning rate (`1.0`) that breaks a well-initialized He network.

But notice the honest row: at a **good** init (He) and a **moderate** learning rate, the plain network
already trains — and is in fact a touch *faster* than the batch-norm version here. So on a small network
like this, batch norm's win is **robustness** — it trains from a bad init and tolerates larger learning
rates — rather than raw speed on an already-healthy net. The dramatic *speed-ups* batch norm is famous
for appear on large, deep vision networks (Ioffe & Szegedy 2015). And a caution worth carrying: the
*reason* batch norm helps is still debated — the original "internal covariate shift" explanation was
challenged by Santurkar et al. (2018), who argue it mainly smooths the loss landscape. What is robust
is the empirical effect you measured here.

## Batch norm's catch, and layer norm

Batch norm has one structural cost: it **couples the samples in a batch**, because each activation is
normalized using the mean and variance of its batch-mates. That is fine for large, independent batches
of images, but it is awkward when the batch is tiny, or when the input is a **sequence** processed one
step at a time — there is no batch to average over.

**Layer normalization** (Ba, Kiros & Hinton 2016) sidesteps this by normalizing across the **features of
each sample** instead of across the batch — so each sample is standardized on its own, independently of
the others. It is the normalization used in recurrent networks and in **transformers**, the architecture
behind today's language models (a horizon we return to in notebook 10). Same idea — keep activations in
a healthy range — but along a different axis.

In [ ]:
# BN standardizes each column (across the batch); LN each row (across the features).
M = np.random.default_rng(0).standard_normal((6, 4)) * 2.0 + 1.0
bn_cols = (M - M.mean(axis=0)) / np.sqrt(M.var(axis=0) + EPS)
ln_rows = (M - M.mean(axis=1, keepdims=True)) / np.sqrt(M.var(axis=1, keepdims=True) + EPS)
print("batch norm — per-feature (column) mean", bn_cols.mean(0).round(2),
      "std", bn_cols.std(0).round(2))
print("layer norm — per-sample  (row)    mean", ln_rows.mean(1).round(2),
      "std", ln_rows.std(1).round(2))

# Why eval needs more than the batch: batch statistics wander from batch to batch.
pop = np.random.default_rng(3).normal(3.0, 2.0, size=5000)   # one feature ~ N(mean 3, var 4)
draw = np.random.default_rng(7)
print("\nbatch statistics wander (feature ~ N(mean 3.0, var 4.0), batch size 32):")
for k in range(6):
    b = draw.choice(pop, size=32, replace=False)
    print(f"  batch {k}: mean {b.mean():.3f}  var {b.var():.3f}")

## What the real `nn.BatchNorm` adds (notebook 8)

The layer you built is the complete forward-and-backward batch-norm transform. The library's
`nn.BatchNorm` adds one piece we left out, visible in the printout above. During **training** it uses the
current batch's mean and variance, exactly as we did. But batch statistics **wander** from batch to
batch, and at **evaluation** time you may have a single sample — no batch to normalize against. So the
library keeps a **running average** of the mean and variance over training and uses *that* at evaluation.

That is why a network with batch norm (or dropout) behaves differently in training and in evaluation,
and why frameworks make you flip a switch — `.train()` versus `.eval()`. We do not build the
running-average machinery here; it arrives, ready-made, as `nn.BatchNorm` in notebook 8. What matters
now is that you know exactly what that one line computes, and what it adds beyond the transform you wrote.

## What you built

- A complete **batch-norm layer** by hand: the **forward** pass (per-feature batch mean/variance →
  standardize → learnable `γ`, `β`) and the **backward** pass (the batch-norm Jacobian), gradient-checked
  to about one part in `10⁹`.
- A demonstration that it **pins every layer's activations to a flat, healthy band regardless of the
  initialization** — small, large, or He all collapse to the same spread across ten layers.
- The **training-time effect**, read honestly: batch norm trains a deep net that had stalled at chance
  and tolerates larger learning rates (robustness), while on an already-healthy net its speed gain here
  is slight.
- **Layer normalization** as the per-sample sibling (the transformer's choice), and what the real
  `nn.BatchNorm` adds — running statistics and the train/eval split.

This **closes the by-hand numpy arc** (notebooks 1–6): you built a deep network from scratch, watched its
gradients vanish and explode (notebook 3), and learned three complementary ways to keep it trainable —
good **initialization** (notebook 4), **dropout** (notebook 5), and **normalization** (here).

## Where this goes next

You have now built, by hand in numpy, everything a modern feed-forward network needs: forward and
backward passes, a training loop, depth, initialization, dropout, and normalization. **Notebook 7 takes
the very same network and rebuilds it in PyTorch** — where `loss.backward()` computes, automatically, the
backward pass you have been writing out by hand. The real `nn.BatchNorm`, `nn.Dropout`, and
`torch.nn.init` then arrive as single lines in notebook 8 — each one a tool whose inner workings you now
understand.

## Your turn

1. **(warm-up)** Sweep the initialization scale — try `small`, `large`, and `he` — and confirm that with
   batch norm the deep net trains from all of them, while without it the poorly-scaled ones stall.
2. **(core)** Implement layer norm as a one-line function, `(z − z.mean(1, keepdims=True)) /
   np.sqrt(z.var(1, keepdims=True) + EPS)`, and check that each *row* comes out with mean 0 and
   variance 1 (compare against the batch-norm column statistics above).
3. **(reach)** Set `γ` to the batch standard deviation and `β` to the batch mean of a layer's
   pre-activations, and confirm with `np.allclose` that the batch-norm output then **recovers** the
   original input — up to the tiny `ε` baked into the normalization (the small residual, about `1e-5`, is
   itself a reminder of why `ε` is there). The learnable parameters really can undo the normalization.

## References

- Ioffe, S., & Szegedy, C. (2015). Batch Normalization: Accelerating Deep Network Training by Reducing
  Internal Covariate Shift. *Proceedings of the 32nd International Conference on Machine Learning* (ICML).
  arXiv:1502.03167.
- Ba, J. L., Kiros, J. R., & Hinton, G. E. (2016). Layer Normalization. arXiv:1607.06450.
- Santurkar, S., Tsipras, D., Ilyas, A., & Madry, A. (2018). How Does Batch Normalization Help
  Optimization? *Advances in Neural Information Processing Systems* (NeurIPS). arXiv:1805.11604.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*, chapter 8. MIT Press.

*Previous:* **12.5 — dropout.**  *Next:* **12.7 — hello-world in PyTorch.**